# T2M Diagnostic Benchmark

## Benchmark Overview

This benchmark evaluates Text-to-Motion (T2M) models at the **atomic requirement level** rather than relying only on overall text-motion alignment or average generation quality.

The benchmark contains **96 predefined prompts** designed to evaluate five motion-generation capabilities:

- **C1 — Basic Action**
- **C2 — Spatial Control**
- **C3 — Attribute Control**
- **C4 — Event Structure**
- **C5 — Composition & Generalisation**

Each prompt is decomposed into atomic requirements such as action execution, movement direction, action count, temporal order, turn direction, attributes, and simultaneity.

The complete benchmark pipeline is:

`Benchmark Definition → T2M Model → Motion Generation → Motion Standardisation → Atomic Requirements → Evaluation Routing → Evidence Extraction → Requirement Evaluation → Diagnostic Analysis`

This notebook implements the benchmark pipeline sequentially so that different T2M models can be evaluated using the same prompts, requirements, and evaluation framework.

## Benchmark Pipeline

### Stage 1 — Benchmark Definition
Load and validate the predefined 96 benchmark prompts and their atomic requirements.

### Stage 2 — Target Model Initialisation
Load the Text-to-Motion model to be evaluated.

### Stage 3 — Motion Generation
Generate motions for the predefined benchmark prompts.

### Stage 4 — Motion Standardisation
Convert model-specific motion outputs into the common benchmark representation.

### Stage 5 — Atomic Requirement Retrieval
Retrieve the predefined atomic requirements associated with each generated motion.

### Stage 6 — Evaluation Routing
Route each atomic requirement to its corresponding evaluator.

### Stage 7 — Evidence Extraction
Extract semantic, geometric, temporal, or kinematic evidence from the standardised motion.

### Stage 8 — Atomic Requirement Evaluation
Compare the observed evidence with the expected requirement and determine PASS / FAIL.

### Stage 9 — Prompt-Level Diagnosis
Identify which requirements failed and classify the corresponding failure types.

### Stage 10 — Capability and Complexity Analysis
Aggregate results to determine where failures emerge as prompt complexity increases.

### Stage 11 — Cross-Model Comparison
Compare diagnostic performance across different T2M models.

# Step 1 — Load and Validate the Benchmark Definition

## Purpose

Load the machine-readable benchmark specification before running any T2M model.

The `benchmark_definition.json` file defines the benchmark independently from any specific T2M model. It contains the 96 predefined prompts together with their metadata and atomic evaluation requirements.

Before motion generation begins, the benchmark definition is validated to ensure that the complete evaluation specification is structurally consistent.

## This step performs

1. Load `benchmark_definition.json`.
2. Verify the benchmark metadata.
3. Confirm that all 96 Prompt IDs are unique.
4. Validate prompt metadata and difficulty labels.
5. Validate all atomic requirements.
6. Confirm that every requirement type is supported by the benchmark evaluation framework.

## Input

`benchmark_definition.json`

## Output

A validated benchmark object containing the complete prompt and requirement specification.

## Step 1.1 — Load the Benchmark Definition

Load the JSON benchmark specification into Python so that prompts, metadata, and atomic requirements can be accessed programmatically.

In [ ]:
import json
from pathlib import Path
from collections import Counter


BENCHMARK_PATH = Path("/content/benchmark_definition.json")


def load_benchmark(path):
    """
    Load the T2M benchmark definition from JSON.
    """

    if not path.exists():
        raise FileNotFoundError(
            f"Benchmark definition not found: {path}"
        )

    with open(path, "r", encoding="utf-8") as f:
        benchmark = json.load(f)

    return benchmark


benchmark = load_benchmark(BENCHMARK_PATH)

print("=" * 60)
print("BENCHMARK DEFINITION LOADED")
print("=" * 60)
print(f"Name:           {benchmark['benchmark_name']}")
print(f"Schema version: {benchmark['schema_version']}")
print(f"Status:         {benchmark['status']}")
print(f"Prompts:        {len(benchmark['prompts'])}")

BENCHMARK DEFINITION LOADED
Name:           T2M_Diagnostic_Benchmark
Schema version: 1.0-draft
Status:         working_definition
Prompts:        96


## Step 1.2 — Define Supported Atomic Requirement Types

The benchmark uses a predefined set of atomic requirement types.

Each requirement type represents a specific property of the generated motion that will later be evaluated by a specialised evaluator.

At this stage, the supported requirement types are defined independently from the evaluator implementation.

In [ ]:
SUPPORTED_REQUIREMENT_TYPES = {
    "action",
    "direction",
    "turn_direction",
    "count",
    "order",
    "attribute",
    "simultaneous",
    "body_side",
    "arm_direction",
    "leg_direction",
    "torso_direction",
    "relation",
    "target",
}

print("Supported requirement types:")
for requirement_type in sorted(SUPPORTED_REQUIREMENT_TYPES):
    print(f"  - {requirement_type}")

Supported requirement types:
  - action
  - arm_direction
  - attribute
  - body_side
  - count
  - direction
  - leg_direction
  - order
  - relation
  - simultaneous
  - target
  - torso_direction
  - turn_direction


## Step 1.3 — Validate the Complete Benchmark Definition

Before running any T2M model, validate the complete benchmark specification.

The validation checks the structural integrity of all prompts and atomic requirements. This prevents invalid benchmark definitions from propagating into motion generation and evaluation.

### Validation checks

- Exactly 96 benchmark prompts are present.
- Every Prompt ID is unique.
- Every prompt contains text.
- Every prompt contains a valid capability.
- Every prompt contains a valid difficulty label.
- Every prompt contains at least one atomic requirement.
- Requirement IDs are unique within each prompt.
- Every requirement contains a valid requirement type.
- Every requirement type is supported by the benchmark.

This validation concerns the **benchmark specification only**. It does not evaluate generated motions.

In [ ]:
VALID_DIFFICULTIES = {
    "Easy",
    "Medium",
    "Hard"
}

VALID_CAPABILITIES = {
    "C1",
    "C2",
    "C3",
    "C4",
    "C5"
}


def validate_benchmark(benchmark):
    """
    Validate the complete benchmark definition.
    """

    prompts = benchmark.get("prompts", [])

    errors = []

    # --------------------------------------------------
    # 1. Validate number of prompts
    # --------------------------------------------------

    if len(prompts) != 96:
        errors.append(
            f"Expected 96 prompts, found {len(prompts)}."
        )

    # --------------------------------------------------
    # 2. Validate Prompt IDs
    # --------------------------------------------------

    prompt_ids = [
        prompt.get("prompt_id")
        for prompt in prompts
    ]

    if len(prompt_ids) != len(set(prompt_ids)):
        errors.append(
            "Duplicate Prompt IDs detected."
        )

    # --------------------------------------------------
    # 3. Validate individual prompts
    # --------------------------------------------------

    total_requirements = 0
    requirement_types = Counter()

    for prompt in prompts:

        prompt_id = prompt.get(
            "prompt_id",
            "<missing>"
        )

        # Prompt text
        if not prompt.get("text"):
            errors.append(
                f"{prompt_id}: Missing prompt text."
            )

        # Capability
        capability = prompt.get("capability")

        if capability not in VALID_CAPABILITIES:
            errors.append(
                f"{prompt_id}: Invalid capability "
                f"'{capability}'."
            )

        # Difficulty
        difficulty = prompt.get("difficulty")

        if difficulty not in VALID_DIFFICULTIES:
            errors.append(
                f"{prompt_id}: Invalid difficulty "
                f"'{difficulty}'."
            )

        # Requirements
        requirements = prompt.get(
            "requirements",
            []
        )

        if not requirements:
            errors.append(
                f"{prompt_id}: No atomic requirements."
            )
            continue

        total_requirements += len(requirements)

        # Requirement IDs
        requirement_ids = [
            req.get("id")
            for req in requirements
        ]

        if len(requirement_ids) != len(
            set(requirement_ids)
        ):
            errors.append(
                f"{prompt_id}: Duplicate requirement IDs."
            )

        # Requirement types
        for requirement in requirements:

            requirement_id = requirement.get(
                "id",
                "<missing>"
            )

            requirement_type = requirement.get(
                "type"
            )

            if requirement_type is None:
                errors.append(
                    f"{prompt_id}/{requirement_id}: "
                    "Missing requirement type."
                )
                continue

            requirement_types[
                requirement_type
            ] += 1

            if (
                requirement_type
                not in SUPPORTED_REQUIREMENT_TYPES
            ):
                errors.append(
                    f"{prompt_id}/{requirement_id}: "
                    f"Unsupported requirement type "
                    f"'{requirement_type}'."
                )

    # --------------------------------------------------
    # Validation Report
    # --------------------------------------------------

    print("=" * 60)
    print("BENCHMARK VALIDATION")
    print("=" * 60)

    print(f"Prompts: {len(prompts)}")
    print(
        f"Unique Prompt IDs: "
        f"{len(set(prompt_ids))}"
    )
    print(
        f"Total Atomic Requirements: "
        f"{total_requirements}"
    )

    print()
    print("Requirement Types:")

    for requirement_type in sorted(
        requirement_types
    ):
        count = requirement_types[
            requirement_type
        ]

        print(
            f"  ✓ {requirement_type:<20} "
            f"{count:>3} requirements"
        )

    print()

    if errors:

        print(
            f"❌ BENCHMARK INVALID "
            f"({len(errors)} errors)"
        )

        for error in errors:
            print(f"  - {error}")

        raise ValueError(
            "Benchmark validation failed."
        )

    print("✅ BENCHMARK VALID")
    print(
        "All prompts and atomic requirements "
        "passed structural validation."
    )

    return True


benchmark_valid = validate_benchmark(
    benchmark
)

BENCHMARK VALIDATION
Prompts: 96
Unique Prompt IDs: 96
Total Atomic Requirements: 367

Requirement Types:
  ✓ action               145 requirements
  ✓ arm_direction         10 requirements
  ✓ attribute             26 requirements
  ✓ body_side             40 requirements
  ✓ count                 28 requirements
  ✓ direction             47 requirements
  ✓ leg_direction          4 requirements
  ✓ order                 28 requirements
  ✓ relation               2 requirements
  ✓ simultaneous          14 requirements
  ✓ target                 2 requirements
  ✓ torso_direction        1 requirements
  ✓ turn_direction        20 requirements

✅ BENCHMARK VALID
All prompts and atomic requirements passed structural validation.


## Step 1.4 — Build the Prompt Index

Create a Prompt ID lookup table for efficient access to benchmark definitions during motion generation and evaluation.

This allows any benchmark prompt to be retrieved directly using its Prompt ID without repeatedly searching through the complete prompt list.

In [ ]:
PROMPT_INDEX = {
    prompt["prompt_id"]: prompt
    for prompt in benchmark["prompts"]
}


def get_prompt_definition(prompt_id):
    """
    Retrieve a benchmark prompt by Prompt ID.
    """

    if prompt_id not in PROMPT_INDEX:
        raise KeyError(
            f"Unknown Prompt ID: {prompt_id}"
        )

    return PROMPT_INDEX[prompt_id]

In [ ]:
example_prompt = get_prompt_definition(
    "C4-15"
)

print("Prompt ID:", example_prompt["prompt_id"])
print("Text:", example_prompt["text"])
print(
    "Requirements:",
    len(example_prompt["requirements"])
)

Prompt ID: C4-15
Text: A person walks forward, jumps once, and then turns right.
Requirements: 6


# Step 2 — Define the T2M Model Interface

## Purpose

Define a model-independent interface between the benchmark framework and the Text-to-Motion (T2M) model being evaluated.

Different T2M models may use different repositories, inference functions, configuration files, and generation procedures. The benchmark should therefore not depend directly on the internal implementation of any specific model.

Instead, every T2M model connected to the benchmark must provide the same basic generation interface:

`Benchmark Prompt → T2M Model → Generated Motion`

The model interface defines the information that the benchmark provides to a T2M model and the information that the model must return.

## Benchmark Input

For each benchmark prompt, the model receives:

- `prompt_id` — unique benchmark Prompt ID
- `text` — text instruction used for motion generation
- `target_frames` — target motion length when required by the model

## Model Output

The model returns a generation result containing:

- `prompt_id`
- `model_name`
- generated motion data
- source motion format
- source frame rate
- additional model-specific metadata when required

The generated motion is not assumed to be standardised at this stage. Model-specific outputs will be converted into the common benchmark motion representation in the subsequent standardisation stage.

This separation allows different T2M models to be connected to the same benchmark without modifying the benchmark evaluation logic.

## Step 2.1 — Define the Model Generation Request

The benchmark converts each predefined prompt into a standard generation request.

A generation request contains the information required to identify the benchmark case and generate its corresponding motion.

This creates a consistent input format regardless of which T2M model is connected to the benchmark.

In [ ]:
from dataclasses import dataclass
from typing import Optional


@dataclass(frozen=True)
class GenerationRequest:
    """
    Standard input passed from the benchmark
    to a Text-to-Motion model.
    """

    prompt_id: str
    text: str
    target_frames: Optional[int] = None

## Step 2.2 — Convert Benchmark Prompts into Generation Requests

Convert the predefined benchmark prompts into model-independent generation requests.

Each benchmark prompt is transformed into a `GenerationRequest` containing its Prompt ID, text instruction, and target frame information.

These requests will later be passed to whichever T2M model is selected for benchmark execution.

In [ ]:
def create_generation_request(prompt_definition):
    """
    Convert one benchmark prompt into
    a standard GenerationRequest.
    """

    return GenerationRequest(
        prompt_id=prompt_definition["prompt_id"],
        text=prompt_definition["text"],
        target_frames=prompt_definition.get(
            "target_frames_20fps"
        )
    )

In [ ]:
GENERATION_REQUESTS = [
    create_generation_request(prompt)
    for prompt in benchmark["prompts"]
]

print("=" * 60)
print("GENERATION REQUESTS CREATED")
print("=" * 60)

print(
    f"Total generation requests: "
    f"{len(GENERATION_REQUESTS)}"
)

GENERATION REQUESTS CREATED
Total generation requests: 96


## Step 2.3 — Inspect a Generation Request

Inspect one generation request to verify that the benchmark prompt has been converted correctly into the standard model input format.

In [ ]:
example_request = next(
    request
    for request in GENERATION_REQUESTS
    if request.prompt_id == "C4-15"
)

print("Prompt ID:", example_request.prompt_id)
print("Text:", example_request.text)
print(
    "Target Frames:",
    example_request.target_frames
)

Prompt ID: C4-15
Text: A person walks forward, jumps once, and then turns right.
Target Frames: 180


## Step 2.4 — Define the Common T2M Model Interface

Define the common interface that every T2M model must implement in order to participate in the benchmark.

The benchmark communicates with a model only through this interface and does not depend on the model's internal inference implementation.

Each connected model must:

1. expose its model name;
2. accept a standard `GenerationRequest`; and
3. return a generation result containing the generated motion and its source-format metadata.

Model-specific implementations will be provided through adapters during pilot testing or final benchmark execution.

In [ ]:
from abc import ABC, abstractmethod


class T2MModelInterface(ABC):
    """
    Common interface for T2M models evaluated
    by the benchmark.
    """

    @property
    @abstractmethod
    def model_name(self):
        """
        Name of the T2M model.
        """
        pass

    @abstractmethod
    def generate(self, request):
        """
        Generate a motion from a GenerationRequest.

        Parameters
        ----------
        request : GenerationRequest
            Standard benchmark generation request.

        Returns
        -------
        GenerationResult
            Model-specific generated motion and
            associated metadata.
        """
        pass

## Step 2.5 — Define the Model Generation Result

Define a standard container for receiving generated motions from different T2M models.

At this stage, the motion data remains in the model's original representation. Information describing the source format is retained so that the subsequent standardisation stage can determine how the motion should be converted.

The generation result therefore separates:

- the generated motion itself;
- benchmark identification information; and
- model-specific representation metadata.

In [ ]:
from dataclasses import dataclass, field
from typing import Any, Dict


@dataclass
class GenerationResult:
    """
    Output returned by a T2M model before
    benchmark motion standardisation.
    """

    prompt_id: str

    model_name: str

    motion: Any

    source_format: str

    source_fps: Optional[float] = None

    metadata: Dict[str, Any] = field(
        default_factory=dict
    )

## Step 2.6 — Validate the Model Input Interface

Validate the complete set of generation requests before connecting an actual T2M model.

This ensures that all 96 benchmark prompts can be passed through the common model interface and that each request contains the information required for model execution.

In [ ]:
def validate_generation_requests(requests):

    errors = []

    prompt_ids = []

    for request in requests:

        prompt_ids.append(
            request.prompt_id
        )

        if not request.prompt_id:
            errors.append(
                "Generation request has no Prompt ID."
            )

        if not request.text:
            errors.append(
                f"{request.prompt_id}: "
                "Missing prompt text."
            )

        if (
            request.target_frames is not None
            and request.target_frames <= 0
        ):
            errors.append(
                f"{request.prompt_id}: "
                "Invalid target frame value."
            )

    if len(prompt_ids) != len(set(prompt_ids)):
        errors.append(
            "Duplicate Prompt IDs detected "
            "in generation requests."
        )

    print("=" * 60)
    print("MODEL INTERFACE VALIDATION")
    print("=" * 60)

    print(
        f"Generation requests: "
        f"{len(requests)}"
    )

    print(
        f"Unique Prompt IDs: "
        f"{len(set(prompt_ids))}"
    )

    if errors:

        print()
        print(
            f"❌ MODEL INTERFACE INVALID "
            f"({len(errors)} errors)"
        )

        for error in errors:
            print(f"  - {error}")

        raise ValueError(
            "Generation request validation failed."
        )

    print()
    print(
        "✅ ALL GENERATION REQUESTS VALID"
    )

    return True


generation_requests_valid = (
    validate_generation_requests(
        GENERATION_REQUESTS
    )
)

MODEL INTERFACE VALIDATION
Generation requests: 96
Unique Prompt IDs: 96

✅ ALL GENERATION REQUESTS VALID


## Step 2 Result

A model-independent interface for Text-to-Motion generation has been established.

The framework now provides:

- **96 standardised generation requests**
- A common input structure for benchmark prompts
- A common interface that any T2M model can implement
- A common container for receiving model-specific generation outputs
- Separation between benchmark logic and model-specific inference code

No specific T2M model has been executed at this stage.

During pilot validation or final benchmark execution, individual T2M models can be connected through model-specific adapters without modifying the benchmark definition or downstream evaluation framework.

**Next:** Define the common motion representation used by the benchmark evaluators.

# Step 3 — Define the Common Motion Representation

## Purpose

Define a model-independent motion representation that will be used as the common input to the benchmark evaluators.

Different Text-to-Motion models may produce motions using different representations, such as joint coordinates, rotations, or model-specific motion features.

The benchmark therefore requires a common representation between model generation and requirement-level evaluation.

`Model-Specific Motion → Standardisation → Common Motion Representation → Evaluators`

The common motion representation stores both the motion data and the metadata required to interpret it consistently.

This allows the same downstream evaluation framework to be applied to motions generated by different T2M models.

## Step 3.1 — Define the Standard Motion Container

The `StandardMotion` structure defines the common motion object used by the benchmark after model-specific standardisation.

The object stores the standardised motion data together with metadata required by downstream evaluators, including frame rate, coordinate system, axis conventions, units, and joint information.

Keeping this information explicitly attached to each motion reduces ambiguity when motions originating from different T2M models are evaluated.

In [ ]:
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional
import numpy as np


@dataclass
class StandardMotion:
    """
    Common motion representation used by
    the benchmark evaluation framework.
    """

    # Benchmark identification
    prompt_id: str
    model_name: str

    # Standardised joint coordinates
    positions: np.ndarray

    # Motion metadata
    fps: float
    coordinate_system: str
    up_axis: str
    forward_axis: str
    units: str

    # Skeleton metadata
    joint_count: int
    joint_names: Optional[List[str]] = None

    # Original representation information
    source_format: Optional[str] = None

    # Additional metadata
    metadata: Dict[str, Any] = field(
        default_factory=dict
    )

## Step 3.2 — Define the Benchmark Coordinate Convention

The benchmark defines a canonical spatial and temporal convention so that motions generated by different T2M models can be interpreted consistently.

All model-specific outputs must be converted to this convention before they are passed to downstream evaluators.

### Benchmark Coordinate Convention

- Coordinate system: Global XYZ
- Coordinate order: X, Y, Z
- Up axis: +Y
- Forward axis: +Z
- Distance unit: metres
- Canonical frame rate: 20 fps

This convention is particularly important for geometry-based evaluators such as movement direction, turn direction, body-side relations, and trajectory analysis.

In [ ]:
BENCHMARK_COORDINATE_CONVENTION = {
    "coordinate_system": "global_xyz",
    "coordinate_order": ("X", "Y", "Z"),
    "up_axis": "+Y",
    "forward_axis": "+Z",
    "units": "metres",
    "fps": 20.0,
}


print("=" * 60)
print("BENCHMARK COORDINATE CONVENTION")
print("=" * 60)

for key, value in BENCHMARK_COORDINATE_CONVENTION.items():
    print(f"{key:<20}: {value}")

BENCHMARK COORDINATE CONVENTION
coordinate_system   : global_xyz
coordinate_order    : ('X', 'Y', 'Z')
up_axis             : +Y
forward_axis        : +Z
units               : metres
fps                 : 20.0


## Step 3.3 — Define the Common Motion Specification

Store the benchmark motion convention explicitly in the framework.

This specification provides a single reference describing the expected representation after motion standardisation.

In [ ]:
COMMON_MOTION_SPEC = {
    "representation": "global_joint_positions",
    "shape": "[T, J, 3]",
    **BENCHMARK_COORDINATE_CONVENTION,
}


print("=" * 60)
print("COMMON MOTION SPECIFICATION")
print("=" * 60)

for key, value in COMMON_MOTION_SPEC.items():
    print(f"{key:<20}: {value}")

COMMON MOTION SPECIFICATION
representation      : global_joint_positions
shape               : [T, J, 3]
coordinate_system   : global_xyz
coordinate_order    : ('X', 'Y', 'Z')
up_axis             : +Y
forward_axis        : +Z
units               : metres
fps                 : 20.0


## Step 3.4 — Define Standard Motion Validation

Before a standardised motion is passed to any evaluator, its representation must be validated.

The validator confirms that the motion follows the common benchmark representation and contains the metadata required for reliable downstream evaluation.

This prevents malformed or incorrectly standardised motions from entering the evaluation pipeline.

In [ ]:
def validate_standard_motion(motion):
    """
    Validate a StandardMotion object before
    requirement-level evaluation.
    """

    errors = []

    positions = motion.positions

    # Check NumPy representation
    if not isinstance(positions, np.ndarray):
        errors.append(
            "positions must be a NumPy array."
        )

    else:

        # Expected [T, J, 3]
        if positions.ndim != 3:
            errors.append(
                "positions must have shape [T, J, 3]."
            )

        elif positions.shape[2] != 3:
            errors.append(
                "The final dimension must contain XYZ coordinates."
            )

        else:

            # Joint count consistency
            if positions.shape[1] != motion.joint_count:
                errors.append(
                    "joint_count does not match "
                    "the motion data."
                )

    # Frame rate
    if motion.fps <= 0:
        errors.append(
            "fps must be greater than zero."
        )

    # Coordinate convention
    if motion.coordinate_system != "global_xyz":
        errors.append(
            "coordinate_system must be 'global_xyz'."
        )

    if motion.up_axis != "+Y":
        errors.append(
            "up_axis must be '+Y'."
        )

    if motion.forward_axis != "+Z":
        errors.append(
            "forward_axis must be '+Z'."
        )

    if motion.units != "metres":
        errors.append(
            "units must be 'metres'."
        )

    if errors:
        raise ValueError(
            "Invalid StandardMotion:\n- "
            + "\n- ".join(errors)
        )

    return True

## Step 3 Result

A model-independent common motion representation has been defined for the benchmark.

All motions passed to downstream evaluators are expected to use:

- Global 3D joint positions
- Shape `[T, J, 3]`
- XYZ coordinate ordering
- `+Y` as the vertical axis
- `+Z` as the forward axis
- Metres as the distance unit
- 20 fps as the canonical benchmark frame rate

The `StandardMotion` structure also retains Prompt ID, model identity, skeleton information, and source-format metadata.

A validation function has been defined to prevent incorrectly standardised motions from entering the evaluation pipeline.

No model-specific motion has been converted at this stage.

**Next:** Define how model-specific motion outputs are converted into this common representation.

# Step 4 — Define the Motion Standardisation Interface

## Purpose

Define a model-independent standardisation interface that converts model-specific T2M outputs into the common motion representation defined in Step 3.

Different T2M models may produce different motion representations, skeleton structures, coordinate systems, units, and frame rates.

Before requirement-level evaluation, these differences must be normalised so that all downstream evaluators receive motion data in a consistent format.

The standardisation process follows:

`GenerationResult → Model-Specific Standardiser → StandardMotion → Validation`

Depending on the source model, standardisation may include:

- Conversion to global 3D joint positions
- Joint mapping
- Coordinate-system conversion
- Axis alignment
- Unit conversion
- Frame-rate resampling
- Motion metadata normalisation

The benchmark itself does not assume how a specific model performs these conversions. Instead, each model provides a compatible standardiser that produces the same `StandardMotion` output.

## Step 4.1 — Define the Standardisation Interface

Every model-specific standardiser must implement the same interface.

The standardiser receives a `GenerationResult` containing the original model output and converts it into a `StandardMotion` that follows the benchmark specification defined in Step 3.

In [ ]:
from abc import ABC, abstractmethod


class MotionStandardizerInterface(ABC):
    """
    Common interface for converting model-specific
    motion outputs into StandardMotion objects.
    """

    @abstractmethod
    def standardize(self, generation_result):
        """
        Convert a GenerationResult into StandardMotion.

        Parameters
        ----------
        generation_result : GenerationResult
            Raw model-specific generation output.

        Returns
        -------
        StandardMotion
            Motion converted to the common benchmark
            representation.
        """
        pass

In [ ]:
STANDARDIZER_REGISTRY = {}


def register_standardizer(model_name, standardizer_class):
    """
    Register a model-specific motion standardiser.
    """

    if not issubclass(
        standardizer_class,
        MotionStandardizerInterface
    ):
        raise TypeError(
            "Standardizer must implement "
            "MotionStandardizerInterface."
        )

    STANDARDIZER_REGISTRY[
        model_name
    ] = standardizer_class


def get_standardizer(model_name):
    """
    Retrieve the registered standardiser
    for a T2M model.
    """

    if model_name not in STANDARDIZER_REGISTRY:
        raise KeyError(
            f"No standardizer registered for "
            f"model: {model_name}"
        )

    return STANDARDIZER_REGISTRY[
        model_name
    ]

In [ ]:
print(
    "Registered standardizers:",
    len(STANDARDIZER_REGISTRY)
)

Registered standardizers: 0


## Step 4.3 — Define the Standardisation and Validation Pipeline

Every model-specific output must be standardised and validated before entering the requirement-level evaluation pipeline.

After conversion, the resulting `StandardMotion` is checked against the common benchmark motion specification.

Only successfully validated motions are allowed to proceed to downstream evaluation.

In [ ]:
def standardize_generation_result(
    generation_result
):
    """
    Standardise a model-specific generation result
    and validate the resulting StandardMotion.
    """

    standardizer_class = get_standardizer(
        generation_result.model_name
    )

    standardizer = standardizer_class()

    standard_motion = standardizer.standardize(
        generation_result
    )

    validate_standard_motion(
        standard_motion
    )

    return standard_motion

## Step 4 Result

A model-independent motion standardisation framework has been established.

The framework now provides:

- A common interface for model-specific motion standardisers
- A registry for associating T2M models with their corresponding standardisation procedures
- A standardisation pipeline that converts `GenerationResult` into `StandardMotion`
- Automatic validation of standardised motions before downstream evaluation

No specific T2M model has been registered or executed at this stage.

Model-specific standardisers will be connected during pilot validation or final benchmark execution.

**Next:** Retrieve the atomic requirements associated with each benchmark motion.

# Step 5 — Retrieve and Bind Atomic Requirements

## Purpose

Retrieve the predefined atomic requirements associated with each standardised motion and bind them to the corresponding benchmark case.

Each `StandardMotion` retains its original benchmark `prompt_id`. This identifier is used to retrieve the evaluation requirements defined for the same prompt in `benchmark_definition.json`.

This step connects the two main components of the benchmark:

`Generated Motion + Expected Atomic Requirements`

The generated motion represents the observed model output, while the atomic requirements define what the motion is expected to satisfy.

No PASS / FAIL decision is made at this stage. The purpose of this step is only to prepare the motion and its expected requirements for evaluator routing.

## Step 5.1 — Retrieve Atomic Requirements by Prompt ID

Use the Prompt ID to retrieve the predefined atomic requirements associated with a benchmark case.

Because the Prompt Index was created in Step 1, the framework can directly access the corresponding prompt definition without searching through all 96 prompts.

In [ ]:
def get_atomic_requirements(prompt_id):
    """
    Retrieve the atomic requirements associated
    with a benchmark Prompt ID.
    """

    prompt_definition = get_prompt_definition(
        prompt_id
    )

    requirements = prompt_definition.get(
        "requirements",
        []
    )

    if not requirements:
        raise ValueError(
            f"No atomic requirements found for "
            f"Prompt ID: {prompt_id}"
        )

    return requirements

In [ ]:
example_requirements = get_atomic_requirements(
    "C4-15"
)

print("=" * 60)
print("ATOMIC REQUIREMENTS")
print("=" * 60)

print("Prompt ID: C4-15")
print(
    f"Number of requirements: "
    f"{len(example_requirements)}"
)

print()

for requirement in example_requirements:
    print(
        f"{requirement['id']}: "
        f"{requirement['type']} = "
        f"{requirement['value']}"
    )

ATOMIC REQUIREMENTS
Prompt ID: C4-15
Number of requirements: 6

r1: action = walk
r2: direction = forward
r3: action = jump
r4: count = 1
r5: turn_direction = right
r6: order = ['walk', 'jump', 'turn']


## Step 5.2 — Define the Requirement-Level Evaluation Case

Define a common container that binds a standardised motion to the atomic requirements associated with its benchmark prompt.

An `EvaluationCase` represents one complete benchmark case immediately before requirement-level evaluator routing.

It contains:

- The Prompt ID
- The original benchmark text
- The standardised generated motion
- The predefined atomic requirements

This structure ensures that the observed motion and its expected conditions remain linked throughout the evaluation pipeline.

In [ ]:
from dataclasses import dataclass
from typing import List, Dict, Any


@dataclass
class EvaluationCase:
    """
    One benchmark case prepared for
    requirement-level evaluation.
    """

    prompt_id: str

    prompt_text: str

    motion: StandardMotion

    requirements: List[Dict[str, Any]]

## Step 5.3 — Bind Standardised Motion to Benchmark Requirements

Create an `EvaluationCase` by matching the Prompt ID stored in a `StandardMotion` with the corresponding benchmark definition.

This operation automatically connects the generated model output with the requirements that must later be evaluated.

In [ ]:
def create_evaluation_case(
    standard_motion
):
    """
    Bind a StandardMotion to its benchmark
    prompt and atomic requirements.
    """

    # Ensure that the motion is valid before
    # entering the evaluation pipeline.
    validate_standard_motion(
        standard_motion
    )

    prompt_definition = get_prompt_definition(
        standard_motion.prompt_id
    )

    requirements = get_atomic_requirements(
        standard_motion.prompt_id
    )

    return EvaluationCase(
        prompt_id=standard_motion.prompt_id,
        prompt_text=prompt_definition["text"],
        motion=standard_motion,
        requirements=requirements
    )

## Step 5.4 — Validate the Evaluation Case

Validate the connection between the standardised motion and its benchmark requirements before evaluator routing.

The validation ensures that the Prompt ID is consistent and that the evaluation case contains at least one valid atomic requirement.

In [ ]:
def validate_evaluation_case(case):
    """
    Validate an EvaluationCase before
    evaluator routing.
    """

    errors = []

    # Prompt ID consistency
    if case.motion.prompt_id != case.prompt_id:
        errors.append(
            "Motion Prompt ID does not match "
            "EvaluationCase Prompt ID."
        )

    # Prompt existence
    if case.prompt_id not in PROMPT_INDEX:
        errors.append(
            f"Unknown Prompt ID: {case.prompt_id}"
        )

    # Prompt text
    if not case.prompt_text:
        errors.append(
            "EvaluationCase has no prompt text."
        )

    # Requirements
    if not case.requirements:
        errors.append(
            "EvaluationCase has no atomic requirements."
        )

    # Requirement types
    for requirement in case.requirements:

        requirement_type = requirement.get("type")

        if (
            requirement_type
            not in SUPPORTED_REQUIREMENT_TYPES
        ):
            errors.append(
                f"Unsupported requirement type: "
                f"{requirement_type}"
            )

    if errors:
        raise ValueError(
            "Invalid EvaluationCase:\n- "
            + "\n- ".join(errors)
        )

    return True

## Step 5 Result

The benchmark can now retrieve the atomic requirements associated with any Prompt ID and bind them to a standardised generated motion.

An `EvaluationCase` has been defined as the common unit passed into the requirement-level evaluation stage.

Each evaluation case contains:

- The benchmark Prompt ID
- The original prompt text
- The standardised generated motion
- The corresponding atomic requirements

This establishes the connection between the model output and the expected benchmark behaviour.

No requirement has been scored at this stage.

**Next:** Define the Evaluator Registry that maps each atomic requirement type to the evaluator responsible for analysing it.

# Step 6 — Define the Evaluator Registry

## Purpose

Define the evaluation architecture used to associate each atomic requirement type with the evaluator responsible for analysing it.

Different requirement types require different forms of evidence.

For example:

- Action requirements require semantic action evidence.
- Direction requirements require trajectory evidence.
- Turn-direction requirements require rotational evidence.
- Count and order requirements require detected action-event evidence.
- Attribute requirements require kinematic or comparative evidence.

The Evaluator Registry provides a central mapping between atomic requirement types and their corresponding evaluators.

This allows the benchmark to automatically select the appropriate evaluation method without embedding evaluator-specific logic inside the main benchmark pipeline.

## Step 6.1 — Define the Common Evaluator Interface

All requirement evaluators implement a common interface.

Each evaluator receives:

- A standardised motion
- One atomic requirement
- Optional shared evidence

Each evaluator returns a structured requirement-level evaluation result.

This common interface allows different evaluator implementations to be used within the same benchmark pipeline.

In [ ]:
from abc import ABC, abstractmethod


class EvaluatorInterface(ABC):
    """
    Common interface for all atomic requirement evaluators.
    """

    @abstractmethod
    def evaluate(
        self,
        motion,
        requirement,
        shared_evidence=None
    ):
        """
        Evaluate one atomic requirement.

        Parameters
        ----------
        motion : StandardMotion
            Standardised motion being evaluated.

        requirement : dict
            Atomic requirement definition.

        shared_evidence : optional
            Evidence that may be reused across
            multiple evaluators.

        Returns
        -------
        RequirementEvaluationResult
            Structured requirement-level result.
        """
        pass

## Step 6.2 — Define the Requirement Evaluation Result

Define a standard result structure for atomic requirement evaluation.

Each evaluator reports:

- The requirement being evaluated
- The expected value
- The observed value
- Supporting evidence
- PASS / FAIL status
- Optional failure information

Using a common result structure allows outputs from different evaluators to be aggregated consistently during prompt-level and capability-level diagnosis.

In [ ]:
from dataclasses import dataclass, field
from typing import Any, Dict, Optional


@dataclass
class RequirementEvaluationResult:
    """
    Standard result produced by an atomic
    requirement evaluator.
    """

    requirement_id: str

    requirement_type: str

    expected: Any

    observed: Any

    evidence: Dict[str, Any] = field(
        default_factory=dict
    )

    result: str = "UNASSESSED"

    failure_type: Optional[str] = None

## Step 6.3 — Define the Evaluator Registry

The Evaluator Registry stores the evaluator associated with each supported atomic requirement type.

Evaluators are registered independently from the benchmark prompts. Therefore, all prompts containing the same requirement type can reuse the same evaluation method.

For example:

`direction → TrajectoryEvaluator`

means that every direction requirement in the benchmark can be routed to the same trajectory-based evaluation logic.

In [ ]:
EVALUATOR_REGISTRY = {}


def register_evaluator(
    requirement_type,
    evaluator_class
):
    """
    Register an evaluator for an atomic
    requirement type.
    """

    if (
        requirement_type
        not in SUPPORTED_REQUIREMENT_TYPES
    ):
        raise ValueError(
            f"Unsupported requirement type: "
            f"{requirement_type}"
        )

    if not issubclass(
        evaluator_class,
        EvaluatorInterface
    ):
        raise TypeError(
            "Evaluator must implement "
            "EvaluatorInterface."
        )

    EVALUATOR_REGISTRY[
        requirement_type
    ] = evaluator_class


def get_evaluator(
    requirement_type
):
    """
    Retrieve the evaluator registered for
    an atomic requirement type.
    """

    if requirement_type not in EVALUATOR_REGISTRY:
        raise KeyError(
            f"No evaluator registered for "
            f"requirement type: "
            f"{requirement_type}"
        )

    return EVALUATOR_REGISTRY[
        requirement_type
    ]

In [ ]:
print(
    "Registered evaluators:",
    len(EVALUATOR_REGISTRY)
)

Registered evaluators: 0


## Step 6.4 — Define Evaluator Registry Validation

Validate the evaluator registry to ensure that every registered evaluator corresponds to a supported atomic requirement type and implements the common evaluator interface.

Full requirement coverage will be checked after the concrete evaluators have been implemented and registered.

In [ ]:
def validate_evaluator_registry(
    require_full_coverage=False
):
    """
    Validate the current evaluator registry.
    """

    errors = []

    for (
        requirement_type,
        evaluator_class
    ) in EVALUATOR_REGISTRY.items():

        if (
            requirement_type
            not in SUPPORTED_REQUIREMENT_TYPES
        ):
            errors.append(
                f"Unsupported requirement type "
                f"in registry: {requirement_type}"
            )

        if not issubclass(
            evaluator_class,
            EvaluatorInterface
        ):
            errors.append(
                f"{requirement_type}: evaluator "
                f"does not implement "
                f"EvaluatorInterface."
            )

    if require_full_coverage:

        missing = (
            SUPPORTED_REQUIREMENT_TYPES
            - set(EVALUATOR_REGISTRY.keys())
        )

        for requirement_type in sorted(missing):
            errors.append(
                f"No evaluator registered for: "
                f"{requirement_type}"
            )

    if errors:
        raise ValueError(
            "Invalid Evaluator Registry:\n- "
            + "\n- ".join(errors)
        )

    return True

In [ ]:
validate_evaluator_registry(
    require_full_coverage=False
)

print("✅ Evaluator Registry structure valid")

✅ Evaluator Registry structure valid


## Step 6 Result

A common evaluator architecture and registry mechanism have been defined.

The framework now provides:

- A common interface for all atomic requirement evaluators
- A standard requirement-level result structure
- A central registry for associating requirement types with evaluators
- Support for shared evidence between related evaluators
- Registry validation before benchmark execution

Concrete evaluator implementations have not yet been registered.

They will be implemented and validated before the final benchmark execution.

**Next:** Define the Evaluation Router that automatically routes each atomic requirement to its registered evaluator.

# Step 7 — Define the Evaluation Router

## Purpose

Automatically route each atomic requirement to the evaluator responsible for analysing that requirement type.

An `EvaluationCase` may contain multiple atomic requirements requiring different evaluation methods.

For example:

`action → ActionEvaluator`

`direction → TrajectoryEvaluator`

`count → CountEvaluator`

`order → OrderEvaluator`

The Evaluation Router reads the requirement type and uses the Evaluator Registry defined in Step 6 to identify the appropriate evaluator.

At this stage, the router determines **which evaluator should process each requirement**. The actual PASS / FAIL evaluation logic is implemented separately by the concrete evaluators.

## Step 7.1 — Define the Requirement Routing Entry

Each atomic requirement is represented by a routing entry before evaluation.

The routing entry records the Prompt ID, requirement definition, and evaluator selected by the Evaluator Registry.

This provides a traceable connection between each benchmark requirement and its evaluation method.

In [ ]:
from dataclasses import dataclass
from typing import Dict, Any, Type


@dataclass
class RequirementRoute:
    """
    Routing information for one atomic requirement.
    """

    prompt_id: str

    requirement: Dict[str, Any]

    evaluator_class: Type[EvaluatorInterface]

## Step 7.2 — Route an Atomic Requirement

Route one atomic requirement to its registered evaluator.

The requirement type is used as the routing key. The router itself does not contain evaluator-specific decision logic; it only queries the Evaluator Registry.

This keeps routing logic independent from individual evaluator implementations.

In [ ]:
def route_requirement(
    prompt_id,
    requirement
):
    """
    Route one atomic requirement to its
    registered evaluator.
    """

    requirement_type = requirement.get(
        "type"
    )

    if not requirement_type:
        raise ValueError(
            f"{prompt_id}: requirement has "
            f"no type."
        )

    evaluator_class = get_evaluator(
        requirement_type
    )

    return RequirementRoute(
        prompt_id=prompt_id,
        requirement=requirement,
        evaluator_class=evaluator_class
    )

## Step 7.3 — Route an Evaluation Case

Route all atomic requirements contained in an `EvaluationCase`.

Each requirement is independently matched to the evaluator registered for its requirement type.

The output is a list of routing entries that describes how the complete benchmark case will be evaluated.

In [ ]:
def route_evaluation_case(
    evaluation_case
):
    """
    Route all atomic requirements in one
    EvaluationCase.
    """

    validate_evaluation_case(
        evaluation_case
    )

    routes = []

    for requirement in (
        evaluation_case.requirements
    ):

        route = route_requirement(
            prompt_id=evaluation_case.prompt_id,
            requirement=requirement
        )

        routes.append(route)

    return routes

## Step 7.4 — Preview the Planned Routing

Before concrete evaluator implementations are registered, the routing configuration stored in the benchmark definition can be inspected.

This preview does not execute an evaluator. It verifies the intended mapping between atomic requirement types and evaluator names.

In [ ]:
def preview_requirement_routing(
    prompt_id
):
    """
    Display the planned evaluator routing
    defined in benchmark_definition.json.
    """

    requirements = get_atomic_requirements(
        prompt_id
    )

    planned_registry = benchmark[
        "evaluator_registry"
    ]

    print("=" * 60)
    print("ROUTING PREVIEW")
    print("=" * 60)

    print(f"Prompt ID: {prompt_id}")
    print()

    for requirement in requirements:

        requirement_type = requirement[
            "type"
        ]

        evaluator_name = (
            planned_registry.get(
                requirement_type,
                "UNREGISTERED"
            )
        )

        print(
            f"{requirement['id']:<4} "
            f"{requirement_type:<18} "
            f"→ {evaluator_name}"
        )

In [ ]:
preview_requirement_routing(
    "C4-15"
)

ROUTING PREVIEW
Prompt ID: C4-15

r1   action             → ActionEvaluator
r2   direction          → TrajectoryEvaluator
r3   action             → ActionEvaluator
r4   count              → CountEvaluator
r5   turn_direction     → RotationEvaluator
r6   order              → OrderEvaluator


## Step 7.5 — Validate Planned Routing Coverage

Validate the planned evaluator routing across all benchmark prompts.

Every atomic requirement must have a corresponding evaluator assignment in the benchmark evaluator registry.

This structural validation ensures complete routing coverage before concrete evaluator implementations are introduced.

In [ ]:
def validate_planned_routing():
    """
    Validate planned evaluator routing for
    every atomic requirement in the benchmark.
    """

    planned_registry = benchmark[
        "evaluator_registry"
    ]

    errors = []

    total_requirements = 0

    for prompt in benchmark["prompts"]:

        for requirement in prompt[
            "requirements"
        ]:

            total_requirements += 1

            requirement_type = requirement[
                "type"
            ]

            if (
                requirement_type
                not in planned_registry
            ):
                errors.append(
                    f"{prompt['prompt_id']} / "
                    f"{requirement['id']}: "
                    f"No planned evaluator for "
                    f"{requirement_type}"
                )

    print("=" * 60)
    print("PLANNED ROUTING VALIDATION")
    print("=" * 60)

    print(
        f"Prompts checked: "
        f"{len(benchmark['prompts'])}"
    )

    print(
        f"Atomic requirements checked: "
        f"{total_requirements}"
    )

    if errors:

        print()
        print(
            f"❌ ROUTING INCOMPLETE "
            f"({len(errors)} errors)"
        )

        for error in errors:
            print(f"  - {error}")

        raise ValueError(
            "Planned routing validation failed."
        )

    print()
    print(
        "✅ ALL REQUIREMENTS HAVE "
        "A PLANNED EVALUATOR"
    )

    return True


planned_routing_valid = (
    validate_planned_routing()
)

PLANNED ROUTING VALIDATION
Prompts checked: 96
Atomic requirements checked: 367

✅ ALL REQUIREMENTS HAVE A PLANNED EVALUATOR


## Step 7 Result

The Evaluation Router has been defined.

The framework can now:

- Read the type of each atomic requirement
- Identify the evaluator responsible for that requirement
- Route all requirements within an evaluation case
- Preview evaluator assignments before concrete evaluator execution
- Validate evaluator-routing coverage across the complete benchmark

All 96 benchmark prompts and their atomic requirements have planned evaluator assignments.

The router itself does not determine PASS / FAIL outcomes. It only determines which evaluator is responsible for each requirement.

**Next:** Implement the concrete evaluators that extract evidence from standardised motions and determine requirement-level PASS / FAIL outcomes.